## Outline

#### The purpose of this script is to run brms models on the CNg data, with nonlinear effects disaggregated by sex 

In [19]:
# Import packages
import os
import mne
import re
import pandas as pd
import numpy as np
import glob
import datetime
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from nilearn import image, datasets
import seaborn as sns
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler

import arviz as az

# Imports for R functionality
from rpy2.robjects.packages import importr
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter

# Explicitly define R functions
base = importr('base')
brms = importr('brms')
loo = importr('loo')
tidybayes = importr("tidybayes")
dplyr = importr("dplyr")

In [20]:
# Write out data?
writeOK=True

In [21]:
# Read config file and define paths
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']

# Update model dir for the appropriate training run (the one with the lowest variational free energy)
training_run = config['analysis_params']['selected_training_run']
model_dir = f'{model_dir}_run{training_run}'

# Define directory containing posthoc output
posthoc_dir = os.path.join(model_dir, 'posthoc')
os.makedirs(posthoc_dir, exist_ok=True)

# Define directory containing brms output
brm_dir = os.path.join(model_dir, 'brms_fits')
os.makedirs(brm_dir, exist_ok=True)

In [22]:
# Plotting params
colours = config['misc']['colours']
n_modes = 6

# Other params
orig_mode_order = config['posthoc']['orig_mode_order']
exclude = config['misc']['exclusion_col']
groups = config['misc']['groups']

orig_mode_order = config['posthoc']['orig_mode_order']
new_mode_order = config['posthoc']['new_mode_order']

now = datetime.datetime.now()
datetime_str = now.strftime("%Y-%m-%d_%H%M")

In [23]:
# Define a helper function to pull the most recent version of a file
# Note that this selects files based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file

In [24]:
# Files to load
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt')) # this is correct for the new analysis
data_fname = recent_fname(os.path.join(posthoc_dir, '*_CNg_coh.csv'))

# Load list of subjects that went into DyNeMo
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

# Read coh values for each mode
df_coh = pd.read_csv(data_fname)

# Sex is sometimes coded as 'F   '. Fix this. 
df_coh['Sex'] = df_coh['Sex'].str.strip()

# compute mean mode amplitudes per subject, for plotting. 
df_coh_avg = df_coh.groupby(['SubjectID','Sex','Dx', 'UniqueID'], as_index=False).mean()

# Define functions

In [25]:
# Function to get peak (i.e., mode) posterior probability, using Arviz kde functionality
def get_az_mode(draws):

    # Ensure we are working with a clean numpy array
    x = np.asarray(draws)
    
    # az.kde returns (grid_points, density_values)
    grid, pdf = az.kde(x)
    
    # The mode is the grid point where density is highest
    return grid[np.argmax(pdf)]

In [26]:
# Function for estimating slope derivative at a given age (in z space), seperatly for M and F
def get_slope_at_age_z(fit, age_z, sex, delta=0.01):

    # delta is in real age units, so convert to z space
    delta_z = delta/age_sd

    post_plus = brms.posterior_smooths(fit, smooth = "s(Age_z, by=Sex)", 
                                       newdata=ro.DataFrame({
                                           'Age_z': ro.FloatVector([age_z + delta_z]), 'Sex': ro.StrVector([sex])}), 
                                       re_formula = ro.r('NA'))
    
    post_minus = brms.posterior_smooths(fit, smooth = "s(Age_z, by=Sex)", 
                                       newdata=ro.DataFrame({
                                           'Age_z': ro.FloatVector([age_z - delta_z]), 'Sex': ro.StrVector([sex])}), 
                                       re_formula = ro.r('NA'))


    # Convert R matrices to NumPy arrays
    with localconverter(default_converter + pandas2ri.converter):
        post_plus_np = np.array(ro.conversion.rpy2py(post_plus))  
        post_minus_np = np.array(ro.conversion.rpy2py(post_minus))

    # Compute finite difference
    slope_post = (post_plus_np - post_minus_np) / (2 * delta_z)

    return slope_post

In [27]:
# Function to compute posterior distributions of slope derivatives across a set of pre-defined age bands, 
# and return summary stats (posterior mode and HDIs) for each band. This outputs seperate results for M and F. 
def compute_band_posterior(fit, age_bands, mode_sd, age_sd):
    
    # Store posteriors for all bands and sexes
    band_posteriors = []

    # Loop through M and F
    for sex in ['M','F']:
        
        # Loop through each age band
        for band_name, ages in age_bands.items():
            
            # Store slopes for all ages within this band
            slopes_list = []

            # Estimate slope for each age in this band
            for age in ages: 

                # Convert age to age_z
                age_z = (age - age_mean) / age_sd
                
                # Compute (a posterior distribution of) local slope derivative for each age
                slope_post = get_slope_at_age_z(fit, age_z, sex) 

                # Back-transform the slope distribution to original scale
                slope_post_back = slope_post * (mode_sd / age_sd)

                # Append the posterior samples
                slopes_list.append(slope_post_back.flatten())

            # Take the average (across ages) draws across the band
            slopes_array = np.vstack(slopes_list)     # shape: (n_ages, n_draws)
            band_slopes = slopes_array.mean(axis=0)   # average across ages for each posterior draw

            # Create DataFrame for this band and sex
            df_band = pd.DataFrame({
                "Band": band_name,
                "Sex": sex,
                "Slope": band_slopes
            })
            
            # Append the dataframe for this band/sex combination
            band_posteriors.append(df_band)

    # Combine all bands and sexes into one DataFrame
    posterior_df = pd.concat(band_posteriors, ignore_index=True)

    # Ensure correct ordering of bands
    posterior_df['Band'] = pd.Categorical(posterior_df['Band'], 
                                           categories=age_bands.keys(), 
                                           ordered=True)

    # Create a summary df with mode and HDI for each band and sex combination
    bands_summary = (
    posterior_df.groupby(['Band', 'Sex'], observed=False).agg(
        mode_prob=('Slope', get_az_mode),  # note that this calls our custom function to get peak probability
        hdi_lower=('Slope', lambda x: az.hdi(x.to_numpy(), hdi_prob=0.95)[0]),
        hdi_upper=('Slope', lambda x: az.hdi(x.to_numpy(), hdi_prob=0.95)[1])
    ).reset_index()
    )

    return posterior_df, bands_summary


In [28]:
# Function to compute M-F difference in slope posteriors for each age band
def compute_sex_differences(posterior_df):

    diffs = []

    # Loop through each band
    for band in posterior_df['Band'].cat.categories:

        # Extract M and F posteriors for this band
        post_M = posterior_df[(posterior_df['Band']==band) & (posterior_df['Sex']=='M')]['Slope'].values
        post_F = posterior_df[(posterior_df['Band']==band) & (posterior_df['Sex']=='F')]['Slope'].values

        # Safety check — they must have the same number of draws
        if len(post_M) != len(post_F):
            raise ValueError(f"Draw mismatch in band {band}: M={len(post_M)}, F={len(post_F)}")

        # Posterior difference (M − F) for each draw
        diff_draws = post_M - post_F

        df_diff = pd.DataFrame({
            "Band": band,
            "SexDifference": diff_draws
        })

        diffs.append(df_diff)

    # Combine all bands
    diffs_df = pd.concat(diffs, ignore_index=True)

    # Summaries (HDI, MAP, mean, etc.)
    diff_summary = (
        diffs_df.groupby(["Band"], observed=False).agg(
            hdi_lower=("SexDifference", lambda x: az.hdi(x.to_numpy(), hdi_prob=0.95)[0]),
            hdi_upper=("SexDifference", lambda x: az.hdi(x.to_numpy(), hdi_prob=0.95)[1]),
            prob_diff=("SexDifference", get_az_mode)
        )
        .reset_index()
    )

    return diffs_df, diff_summary

# Set up for brms

In [29]:
# Z-transform Age and mode amplitude (we'll only model the z-scored data)
df_coh['Age_z'] = zscore(df_coh['Age'])
df_coh['mode1_z'] = zscore(df_coh['mode1_coh'])
df_coh['mode2_z'] = zscore(df_coh['mode2_coh'])
df_coh['mode3_z'] = zscore(df_coh['mode3_coh'])
df_coh['mode4_z'] = zscore(df_coh['mode4_coh'])
df_coh['mode5_z'] = zscore(df_coh['mode5_coh'])
df_coh['mode6_z'] = zscore(df_coh['mode6_coh'])

# Get original mean and SD for age 
age_mean = df_coh['Age'].mean()
age_sd = df_coh['Age'].std()

# Assign to R environment for brms
with localconverter(default_converter + pandas2ri.converter):
    r_df_coh = ro.conversion.py2rpy(df_coh)
ro.globalenv["df"] = r_df_coh

# Ensure sex is treated as a factor
ro.r('df$Sex <- factor(df$Sex)')

In [ ]:
# Fitting params 
n_chains = 4
n_cores = 4
n_iter = 20000  # should be 20,000 for final analysis
n_warmup = 5000  # should be 5,000 for final analysis
control = ro.ListVector({"adapt_delta": 0.99})

# Define desired probability for HDIs
prob=0.95

# Define priors
priors = ro.r(
    'c('
        'brms::prior("normal(0, 1)", class = "Intercept"),'
        'brms::prior("normal(0, 1)", class = "b"),'
        'brms::prior("normal(0, 0.25)", class = "sds"),'
        'brms::prior("normal(0, 1)", class = "sigma")'
    ')'
)

# Define age groups for which we will later estimate HDIs on smooth terms
age_bands = {'5-9': range(5, 9),
             '10-14': range(10, 14),
             '15-19': range(15, 19),
             '20-24': range(20, 24),
             '25-29': range(25, 29),
             '30-34': range(30, 34),
             '35-40': range(35, 40)
             }

# Estimate parameters for each mode

In [ ]:
# Fit a brms model for each mode
fits = []

for m in range(1,7):

    print(m)

    # Print out mode #
    print(f'Running model for {orig_mode_order[m-1]} Mode')
    
    # Fit the model
    fit = brms.brm(
            ro.Formula(f"mode{m}_z ~ Sex + s(Age_z, by=Sex)"),
            data=r_df_coh,
            prior=priors, 
            sample_prior="yes",
            chains=n_chains, iter=n_iter, warmup=n_warmup, cores=n_cores,
            control = control, backend = "cmdstan", refresh=5000,
        )

    fits.append(fit)

    # Save the model and model summary to disk
    if writeOK:
        fit_fname = os.path.join(brm_dir, f'{datetime_str}_brms_mode{m}_coh_by_smooth_Age_disagg_by_Sex_Zspace')
        base.saveRDS(fit, file=f'{fit_fname}.rds')

        # Save the model summary to txt
        summary = str(brms.print_brmsfit(fit))
        with open(f'{fit_fname}_summary.txt', 'w') as f:
            f.write(summary)




In [30]:
# Optional: Read the pre-computed models from disk
fits = []
for i in range(6):
    fit_fname = os.path.join(brm_dir,
                             recent_fname(os.path.join(brm_dir, f"*_brms_mode{i+1}_coh_by_smooth_Age_disagg_by_Sex_Zspace.rds")))
    fit = base.readRDS(fit_fname)
    fits.append(fit)  

In [ ]:
# Loop through fits, getting smooths for age, and back-transforming to original space
all_smooths = []

for f, fit in enumerate(fits):

    print(f'Mode {f}')


    # Get mode mean and SD for back-transformation
    mode_mean = df_coh[f'mode{f+1}_coh'].mean()
    mode_sd = df_coh[f'mode{f+1}_coh'].std()

    # Use tidybayes in R to get HDIs
    ro.globalenv['current_fit'] = fit

    # Annoyingly I just cannot get the following to work with simple rpy2 functionality. So, I'll use pure 
    # R code for each step, and assign the resulting variable to the global R environment (so that the next step
    # can use it). Frustrating.
    
    # Extract variables (age and sex) ffrom the model
    age_z = fit.rx2("data").rx2("Age_z")
    sex = fit.rx2("data").rx2("Sex")
    
    # Create age grid for predictions in z space
    # age_grid = ro.DataFrame({
    #     "Age_z": base.seq(base.min(age_z), base.max(age_z), **{"length.out": 100})
    #     })

    age_grid = base.seq(base.min(age_z), base.max(age_z), **{"length.out": 100})
    
    # Get levels of sex and combine with age grid 
    sex_levels = base.unique(sex)

    # Create expand.grid equivalent
    grid = base.expand_grid(
        Age_z=age_grid,
        Sex=sex_levels
    )

    ro.globalenv["grid"] = grid

    # Get posterior draws
    draws =  ro.r('add_epred_draws(current_fit, newdata = grid, re_formula = NA)')
    ro.globalenv["draws"] = draws
    
    # Compute HDIs along the smooth term. tidybayes::mode_hdi should automatically groups by columns in 'grid' (Age_z and Sex)
    hdi_results = ro.r('mode_hdi(draws, .epred, .width = 0.95)')
    ro.globalenv["hdi_results"] = hdi_results

    # Summarize as a dataframe containing mode, upper HDI limit, lower HDI limit (for every step in the age grid, seperately for M and F)
    hdi_summary = ro.r('''
        data.frame(
            Sex = hdi_results$Sex,
            Age_z = hdi_results$Age_z,
            estimate__ = hdi_results$.epred,
            lower__ = hdi_results$.lower,
            upper__ = hdi_results$.upper
        )
        ''')
    

    # Convert to pandas
    with localconverter(default_converter + pandas2ri.converter):
        smooths_df = ro.conversion.rpy2py(hdi_summary)

    # Rename columns to indicate Z-space
    smooths_df = smooths_df.rename(columns={
        'estimate__': 'estimate__z',
        'lower__': 'lower__z',
        'upper__': 'upper__z'
    })  

    # Back-transform to original scale
    smooths_df['Age'] = (smooths_df['Age_z'] * age_sd) + age_mean  
    smooths_df['estimate__'] = (smooths_df['estimate__z'] * mode_sd) + mode_mean 
    smooths_df['lower__'] = (smooths_df['lower__z'] * mode_sd) + mode_mean
    smooths_df['upper__'] = (smooths_df['upper__z'] * mode_sd) + mode_mean 

    all_smooths.append(smooths_df)

In [ ]:
# Get band posteriors and HDIs for each mode (this will take a few minutes)
posterior_band_dfs = []
band_summaries = []
for mode in range(len(fits)):

    # Get mode SD for back-transformation
    mode_mean = df_coh[f'mode{f+1}_coh'].mean()
    mode_sd = df_coh[f'mode{f+1}_coh'].std()

    posterior_df, band_summary = compute_band_posterior(fits[mode], age_bands, mode_sd=mode_sd, age_sd=age_sd)

    posterior_band_dfs.append(posterior_df)
    band_summaries.append(band_summary)


In [ ]:
# Get sex differences for each mode
sex_diff_summaries = []

for mode in range(n_modes):
    _, diff_summary = compute_sex_differences(posterior_band_dfs[mode])
    sex_diff_summaries.append(diff_summary)

# Plot results

In [ ]:
# Define the oredr in which we'd like to plot modes
mode_order = [orig_mode_order.index(name) for name in new_mode_order]

# Reorder colours if needed
colours_new_order = [colours[i] for i in mode_order]


colours_FM = ["#E64B35",  # reddish-orange for Female
              "#4DBBD5"]  # turquoise/blue for Male

In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(15, 12))

# Loop through axes/modes
for plot_idx, mode in enumerate(mode_order):
    # Plot CNg amplitude in mode 1 by age
    #sns.regplot(data=df_coh_avg, x='Age', y=f'mode{mode+1}', ax=ax)  # mode is zero-indexed
    # ax.set_xlabel('Age (years)')
    # ax.set_ylabel('Mean CNg Amplitude')
    # ax.set_title(f'Correct Nogo Amplitude vs Age\{orig_mode_names[mode]}')

    #  Get the fitted model for this mode
    fit = fits[mode]

    # Plot smooth age-amplitude curves in the left column...
    ax_left = axes[plot_idx, 0]
    ax_left.set_title(orig_mode_order[mode])

    # and kde plots in the center (F), right (M), and difference (M-F)
    ax_cent = axes[plot_idx, 1]
    ax_right = axes[plot_idx, 2]

    ax_diff = axes[plot_idx, 3]

    ax_cent.set_title('Female')
    ax_right.set_title('Male')

    # Get the smooths and band summaries for this mode
    smooth_df = all_smooths[mode]
    band_summary = band_summaries[mode]


    # Get raw column for this mode
    mode_col = f'mode{mode+1}_coh'

    # Define vertical offsets for plotting HDIs
    y_offsets = np.arange(len(age_bands))

    # Loop through Sex
    for s, sex in enumerate(['F', 'M']):

        # Select data for this sex
        smooth_df_s = smooth_df[smooth_df['Sex']==sex]
        band_summary_s = band_summary[band_summary['Sex']==sex]
        raw_df = df_coh_avg[df_coh_avg['Sex']==sex]

        # Plot the curve in the left column
        ax_left.plot(
            smooth_df_s['Age'], 
            smooth_df_s['estimate__'], 
            color=colours_FM[s], 
            linewidth=2, 
            label=sex
        )

        # # And fill between the HDI
        ax_left.fill_between(
            smooth_df_s['Age'], 
            smooth_df_s['lower__'], 
            smooth_df_s['upper__'], 
            color=colours_FM[s], 
            alpha=0.5, 
            label=f'{int(prob*100)}% HDI'
        )

        # Plot the original datapoints
        ax_left.scatter(
            raw_df['Age'], 
            raw_df[mode_col],   # Plotting the same mode in each loop for debugging!
            color=colours_FM[s], 
            s=10, 
            alpha=0.8
        )

        # # add a regplot for sanity checking
        # sns.regplot(
        #     data=df_coh_avg[df_coh_avg['Sex']==sex], 
        #     x='Age', 
        #     y=mode_col, 
        #     scatter=False, 
        #     scatter_kws={'s': 10, 'alpha': 0.5, 'color': 'blue'},
        #     line_kws={'color': 'blue', 'alpha': 0.3, 'linestyle': '--'},
        #     ax=ax_left
        # )

        # A few outliers are really distorting  the y axis. Let's set the upper y axis limit to 
        # the 95th percentile of Y values
        ymin, ymax = np.percentile(df_coh_avg[mode_col], [5, 95])
        ax_left.set_ylim(ymin, ymax) 

        # add x label
        if plot_idx == len(fits)-1:
            ax_left.set_xlabel('Age')
        else:
            ax_left.set_xlabel('')
            ax_left.set_xticklabels([])
        
        # Plot HDIs

        # Select axis (center column for F, right column for M)
        if sex=='F':
            ax = ax_cent
        else:
            ax = ax_right

        for row_idx, (_, row) in enumerate(band_summary_s.iterrows()):

            # Add MAP
            ax.plot(row['mode_prob'], y_offsets[row_idx], marker='d', color=colours_FM[s])

            # Add 95% HDI
            ax.hlines(
                y=y_offsets[row_idx],
                xmin=row['hdi_lower'],
                xmax=row['hdi_upper'],
                color=colours_FM[s],
                linewidth=2
            )

        ax.set_ylabel('Age Band')
        ax.set_xlim([-0.002, 0.002])
        ax.set_yticks(y_offsets)
        ax.set_yticklabels(age_bands.keys())
        ax.axvline(x=0, color='grey', linestyle='--', linewidth=1)

        if plot_idx == len(fits)-1:
            ax.set_xlabel('Slope Derivative')
        else:
            ax.set_xlabel('')
            ax.set_xticklabels([])

    # Finally, plot HDI for the sex difference in the far right column
    bands_sex_diff_summary = sex_diff_summaries[mode]

    for row_idx, (_, row) in enumerate(bands_sex_diff_summary.iterrows()):
        ax_diff.plot(row['prob_diff'], y_offsets[row_idx], marker='d', color='grey')
        ax_diff.hlines(
            y=y_offsets[row_idx],
            xmin=row['hdi_lower'],
            xmax=row['hdi_upper'],
            color='grey',
            linewidth=2
        )

    ax_diff.set_ylabel('Age Band')
    ax_diff.set_title('M - F Difference')
    ax_diff.set_xlim([-0.002, 0.002])
    ax_diff.set_yticks(y_offsets)
    ax_diff.set_yticklabels(age_bands.keys())
    ax_diff.axvline(x=0, color='grey', linestyle='--', linewidth=1)

    if plot_idx == len(fits)-1:
        ax_diff.set_xlabel('Slope Derivative')
    else:
        ax_diff.set_xlabel('')
        ax_diff.set_xticklabels([])

plt.tight_layout()

# Save out
if writeOK:
    plt.savefig(os.path.join(model_dir, 'figures', f'mode_COH_age_smooths_and_band_HDI_summary_sexdiffs.png'), dpi=300)
plt.show()

#plt.close()

# Formal model comparisons with LOO

In [ ]:
# Load fits without sex disaggregation
fits_withsex = fits
fits_nosex = []

for i in range(6):
    fit_fname = os.path.join(brm_dir,
                             recent_fname(os.path.join(brm_dir, f"*_brms_mode{i+1}_coh_by_smooth_Age_Zspace.rds")))
    fit = base.readRDS(fit_fname)
    fits_nosex.append(fit)



In [ ]:
# Do model comparisons. Getting this to work with rpy2 is overly complicated, so we'll just use an R code block instead
ro.globalenv["fits_withsex"] = fits_withsex
ro.globalenv["fits_nosex"] = fits_nosex

ro.r('''
library(loo)

for (k in 1:6) {
    cat("Mode", k, "\\n")
    loo_m1 <- loo(fits_withsex[[k]])
    loo_m2 <- loo(fits_nosex[[k]])
    
    comp <- loo_compare(loo_m1, loo_m2)
    
    
    print(comp)
    
    delta_looic <- comp[2, "elpd_diff"] * -2
    cat("Delta LOOIC:", delta_looic, "\\n\\n")
}
''')